# L21 · 函数调用：让 AI 动手做事

**学习目标**
- 理解「Function Calling」：AI 不直接回答，而是「决定调用哪个工具」
- 亲手实现一个「工具路由」：根据语义选择函数执行
- 体会 AI Agent「会调用外部能力」的核心机制

**前置依赖**：L20（Prompt）、L03（函数）、L13（API）  
**预计时长**：45 分钟  
**技术栈**：纯 Python（离线 mock 工具，无需真实 API）

---

## 概念讲解：函数调用 = 给 AI 装上「手」

只聊天的 AI 是「嘴」。真正有用的 AI 需要「手」——调用你的函数去查数据库、发邮件、算数。

流程：
1. 你告诉 AI：「你有三个工具：查天气、计算器、查库存」
2. 用户问：「北京天气？」
3. AI 不直接编答案，而是**返回「调用查天气(北京)」这个决定**
4. 你的程序真正执行该函数，把结果喂回 AI，AI 再组织成自然语言

本课我们用规则模拟第 3 步的「路由决策」。

## 第一步：定义三个「工具函数」

In [ ]:
import random

def get_weather(city):
    return f"{city} 当前 {random.choice(['晴','多云','小雨'])}，{random.randint(15,30)}°C"

def calculator(expr):
    try:
        return str(eval(expr, {"__builtins__": {}}, {}))   # 仅限算术，安全沙箱
    except Exception as e:
        return f"算不了：{e}"

def check_stock(item):
    db = {"苹果": 50, "香蕉": 12, "西瓜": 0}
    n = db.get(item, 0)
    return f"{item} 库存 {n} 件" + ("（缺货）" if n == 0 else "")

print(get_weather("北京"))
print(calculator("23 * 7 + 4"))
print(check_stock("西瓜"))

## 第二步：模拟「AI 路由决策」

In [ ]:
def route(user_text):
    """极简意图识别：决定该调哪个工具（真 LLM 由模型产出 JSON 决定）"""
    t = user_text
    if "天气" in t:
        city = next((c for c in ["北京","上海","广州","深圳"] if c in t), "北京")
        return get_weather, [city], f"调用 get_weather({city})"
    if any(op in t for op in ["+", "-", "*", "/", "算", "等于"]):
        expr = "".join(ch for ch in t if ch.isdigit() or ch in "+-*/.()")
        return calculator, [expr], f"调用 calculator({expr})"
    if "库存" in t or "货" in t:
        for item in ["苹果","香蕉","西瓜"]:
            if item in t:
                return check_stock, [item], f"调用 check_stock({item})"
    return None, [], "无法匹配工具，转交通用回答"

for q in ["北京天气怎么样", "帮我算 12*8", "西瓜还有库存吗"]:
    fn, args, log = route(q)
    result = fn(*args) if fn else "我是助手，但这个问题我暂时没有对应工具。"
    print(f"问：{q}\n  → {log}\n  → 结果：{result}\n")

# 🎯 AHA 顿悟单元格：你的「会动手的 AI 助手」

运行下面代码。你会得到一个**交互式助手**：输入不同需求，它会自动「判断该调哪个工具」并执行，
最后用自然语言汇报。试着改 `questions` 列表，加一句你自己的需求。

> 你刚刚搭出了 AI Agent 的骨架：感知意图 → 选择工具 → 执行 → 汇报。
> 今天最火的「AI 帮我在网页下单/查数据库/画图」，核心就是这个循环。

In [ ]:
# ===== 运行我！看 AI 如何自动选工具并执行 =====
questions = [
    "北京天气怎么样",
    "帮我算一下 99 * 7",
    "苹果还有货吗",
    "上海今天天气",
]
print("  🤖 会动手的 AI 助手已上线（自动路由工具）\n")
for q in questions:
    fn, args, log = route(q)
    result = fn(*args) if fn else "（无对应工具，转人工/通用回答）"
    print(f"  👤 你：{q}")
    print(f"  🔧 助手决策：{log}")
    print(f"  🤖 助手：{result}\n")
print("  ✨ 它能查天气、算算术、查库存 —— 这就是 Function Calling 的魔法！")

# 📝 讲师备课笔记（接手 Agent 专用）

**本课难点**：需讲清「AI 产出的是『调用决策』而非最终答案」这一反转；真 LLM 用 JSON schema 声明工具。  
**易错点**：`eval` 安全风险——本方案用沙箱 `{"__builtins__":{}}` 仅限算术，备课笔记须强调生产不用 eval，真场景用 `ast.literal_eval` 或专用库。  
**AHA 机制**：多工具自动路由+自然语言汇报，强「AI 真的会干活」实感，且离线可跑。  
**衔接**：L22 RAG（另一种「外挂能力」：知识）；L23 Agent（把工具调用做成循环规划）。  
**真 LLM 衔接**：注明 OpenAI `tools=[{type:function,...}]` 参数可替换 `route()`，给出方向不写全码。

# 📚 作业 / 下一步

1. 给 `questions` 加一句「深圳天气」，看路由是否正确。
2. 加一个 `send_email` 工具函数并扩展 `route` 识别「发邮件」。
3. 下一课 **L22 RAG：给 AI 外接大脑** —— 让 AI 回答它「训练时没见过」的私有知识。